# M3.5 · Initialization ve aktivasyonlar

Derin bir ağın ilk adımda bile eğitilebilir olması doğru başlatmaya bağlı. Varyansın katmanlar boyunca nasıl çarpıldığını elle hesaplıyor, saf Python'da altı aktivasyonu ve türevlerini yazıyor, NumPy'da 30 katmanlık ağda aktivasyon istatistiklerini izliyor, PyTorch'ta `nn.init` ve hook'larla katman katman ileri/geri istatistik topluyor ve en sonda kritik başlatma ölçeğini 64 ölçeği birden simüle ederek buluyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m3-5-init.html](https://m-gonul.github.io/pytorch-lab/m3-5-init.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m3_5_init.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    pts = [-3.0, -1.2, -0.3, 0.4, 1.0, 2.5]\n    for name in (\"relu\", \"leaky_relu\", \"sigmoid\", \"tanh\", \"gelu\", \"silu\"):\n        def t(name=name):\n            f, df = need(ns, name), need(ns, \"d_\" + name)\n            h = 1e-6\n            close([df(x) for x in pts], [(f(x + h) - f(x - h)) / (2 * h) for x in pts], rtol=1e-5, atol=1e-7, what=f\"d_{name}\")\n        R.test(f\"d_{name} sayısal türevle aynı\", t)\n    R.test(\"değerler: GELU(1) = 0.8412, SiLU(−2) = −0.2384, sigmoid(−800) = 0\",\n           lambda: close([need(ns, \"gelu\")(1.0), need(ns, \"silu\")(-2.0), need(ns, \"sigmoid\")(-800.0)], [0.8411919906082768, -2 / (1 + math.e ** 2), 0.0], rtol=1e-7, atol=1e-300))\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"activation_stds: yalnız katman döngüsü\", lambda: max_loops(src, 1, func=\"activation_stds\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    R.test(\"xavier_std, kaiming_std\", lambda: close([need(ns, \"xavier_std\")(100, 100), need(ns, \"kaiming_std\")(100), need(ns, \"xavier_std\")(300, 100)], [0.1, 0.02 ** 0.5, 0.005 ** 0.5]))\n\n    def ref(X, depth, width, std, act, seed):\n        rng = np.random.default_rng(seed); H, o = X, []\n        for _ in range(depth):\n            H = act(H @ (rng.standard_normal((width, width)) * std)); o.append(H.std())\n        return o\n    relu = lambda z: np.maximum(0, z)\n    X = np.random.default_rng(0).standard_normal((300, 128))\n    R.test(\"activation_stds referansla aynı\", lambda: close(need(ns, \"activation_stds\")(X, 5, 128, 0.1, np.tanh, np.random.default_rng(2)), ref(X, 5, 128, 0.1, np.tanh, 2)))\n\n    def scen():\n        f = need(ns, \"activation_stds\")\n        rows = []\n        cases = [(\"std = 1, ReLU\", 1.0, relu), (\"Xavier, ReLU\", need(ns, \"xavier_std\")(128, 128), relu),\n                 (\"Kaiming, ReLU\", need(ns, \"kaiming_std\")(128), relu), (\"Xavier, tanh\", need(ns, \"xavier_std\")(128, 128), np.tanh)]\n        res = {}\n        for name, s, a in cases:\n            with np.errstate(over=\"ignore\", invalid=\"ignore\"):\n                o = f(X, 30, 128, s, a, np.random.default_rng(3))\n            res[name] = o\n            rows.append([name, f\"{o[0]:.3g}\", f\"{o[9]:.3g}\", f\"{o[-1]:.3g}\"])\n        R.table(\"Aktivasyon std'si, 30 katman (genişlik 128)\", [\"başlatma\", \"katman 1\", \"katman 10\", \"katman 30\"], rows)\n        k = res[\"Kaiming, ReLU\"]\n        ensure(0.3 < k[-1] / k[0] < 3, f\"Kaiming + ReLU kararlı olmalı: {k[0]:.3g} → {k[-1]:.3g}\")\n        ensure(res[\"Xavier, ReLU\"][-1] < 1e-3 * res[\"Xavier, ReLU\"][0], \"Xavier + ReLU sönmeli\")\n        ensure(not np.isfinite(res[\"std = 1, ReLU\"][-1]) or res[\"std = 1, ReLU\"][-1] > 1e10, \"std = 1 patlamalı\")\n    R.test(\"Kaiming + ReLU kararlı, Xavier + ReLU sönüyor, std = 1 patlıyor\", scen)\n",
 "d": "import torch\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef deep(width=128, depth=20, act=nn.ReLU):\n    L = []\n    for _ in range(depth):\n        L += [nn.Linear(width, width), act()]\n    return nn.Sequential(*L)\n\n\ndef check(ns, R):\n    def init():\n        torch.manual_seed(0)\n        m = need(ns, \"init_model\")(nn.Sequential(nn.Linear(500, 400), nn.ReLU(), nn.Linear(400, 300)), \"kaiming\")\n        ensure(abs(m[0].weight.detach().std().item() / (2 / 500) ** 0.5 - 1) < 0.05, f\"kaiming std {m[0].weight.detach().std().item():.4f}\")\n        ensure(float(m[0].bias.detach().abs().max()) == 0, \"bias'lar 0 olmalı\")\n        m = need(ns, \"init_model\")(m, \"xavier\")\n        ensure(abs(m[2].weight.detach().std().item() / (2 / 700) ** 0.5 - 1) < 0.05, f\"xavier std {m[2].weight.detach().std().item():.4f}\")\n    R.test(\"init_model: doğru std'ler, sıfır bias\", init)\n\n    def stats():\n        torch.manual_seed(0)\n        X = torch.randn(256, 128)\n        rows = []\n        res = {}\n        for scheme, act in ((\"kaiming\", nn.ReLU), (\"xavier\", nn.ReLU), (\"xavier\", nn.Tanh)):\n            torch.manual_seed(1)\n            m = need(ns, \"init_model\")(deep(act=act), scheme)\n            a, g = need(ns, \"layer_stats\")(m, X)\n            ensure(len(a) == 20 and len(g) == 20, f\"20 değer bekleniyordu, {len(a)} ve {len(g)} geldi\")\n            res[(scheme, act.__name__)] = (a, g)\n            rows.append([f\"{scheme} + {act.__name__}\", f\"{a[0]:.3g}\", f\"{a[-1]:.3g}\", f\"{g[0]:.3g}\", f\"{g[-1]:.3g}\"])\n        R.table(\"20 katman: aktivasyon ve gradyan std'si\", [\"kurulum\", \"akt. ilk\", \"akt. son\", \"grad ilk\", \"grad son\"], rows)\n        a, g = res[(\"kaiming\", \"ReLU\")]\n        ensure(0.2 < a[-1] / a[0] < 5, \"Kaiming + ReLU aktivasyonları kararlı olmalı\")\n        a2, _ = res[(\"xavier\", \"ReLU\")]\n        ensure(a2[-1] < 0.05 * a2[0], \"Xavier + ReLU aktivasyonları sönmeli\")\n    R.test(\"layer_stats: 20 katman, beklenen davranışlar\", stats)\n\n    def clean():\n        torch.manual_seed(0)\n        m = need(ns, \"init_model\")(deep(depth=3), \"kaiming\")\n        need(ns, \"layer_stats\")(m, torch.randn(8, 128))\n        ensure(all(len(x._forward_hooks) == 0 for x in m.modules()), \"hook'lar kaldırılmamış\")\n    R.test(\"hook'lar sonunda kaldırılıyor\", clean)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef one(X, Gs, s, act):\n    H, o = X, []\n    for G in Gs:\n        H = act(s / X.shape[1] ** 0.5 * (H @ G.T)); o.append(H.std())\n    return torch.stack(o)\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız katman döngüsü\", lambda: max_loops(src, 1, func=\"std_vs_scale\", why=\"ölçekler üzerinde döngü olmamalı\"))\n    g = torch.Generator().manual_seed(0)\n    X = torch.randn(128, 64, generator=g, dtype=torch.float64)\n    Gs = torch.randn(30, 64, 64, generator=g, dtype=torch.float64)\n    scales = torch.linspace(0.6, 2.4, 19, dtype=torch.float64)\n\n    def eq():\n        out = need(ns, \"std_vs_scale\")(X, Gs, scales, torch.relu)\n        ensure(tuple(out.shape) == (19, 30), f\"shape {tuple(out.shape)}\")\n        for i in (0, 9, 18):\n            close(out[i], one(X, Gs, float(scales[i]), torch.relu), rtol=1e-9, what=f\"s = {float(scales[i]):.2f}\")\n        ratio = out[:, -1] / X.std()\n        k = int((ratio.log().abs()).argmin())\n        return f\"30 katmandan sonra std'yi en iyi koruyan ölçek s = {float(scales[k]):.2f} (teori √2 ≈ 1.41)\"\n    R.test(\"tek tek simülasyonla aynı; kritik ölçek ≈ √2\", eq)\n    if \"std_vs_scale\" not in ns:\n        return\n    Xf, Gf = X.float(), Gs.float()\n    sc = torch.linspace(0.5, 2.5, 64)\n    t_loop = bench(lambda: [one(Xf, Gf, float(s), torch.relu) for s in sc], repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"std_vs_scale\"](Xf, Gf, sc, torch.relu), repeat=3, number=1)\n    R.table(\"64 ölçek × 30 katman\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for s: simülasyon\", fmt_time(t_loop), \"1×\"], [\"senin std_vs_scale\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}× hızlı\"]])\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        close(need(ns, \"selu\")(np.array([1.0, -1.0])), [1.0507009873554805, 1.0507009873554805 * 1.6732632423543772 * (np.exp(-1) - 1)], what=\"SELU\")\n        m, s2 = need(ns, \"gauss_moments\")(ns[\"selu\"])\n        close((m, s2), (0.0, 1.0), atol=1e-4, what=\"sabit nokta: E[selu(z)] = 0, E[selu(z)²] = 1\")\n        mr, s2r = ns[\"gauss_moments\"](lambda z: np.maximum(z, 0))\n        close((mr, s2r), (1 / np.sqrt(2 * np.pi), 0.5), atol=1e-5, what=\"ReLU: ortalama 1/√(2π), ikinci moment 1/2\")\n        rows = []\n        cfg = ((\"SELU + LeCun\", ns[\"selu\"], \"lecun\"), (\"SELU + He\", ns[\"selu\"], \"he\"), (\"ELU + LeCun\", lambda z: np.where(z > 0, z, np.exp(np.minimum(z, 0)) - 1), \"lecun\"),\n               (\"ReLU + He\", lambda z: np.maximum(z, 0), \"he\"), (\"tanh + Glorot\", np.tanh, \"glorot\"))\n        res = {}\n        for name, f, init in cfg:\n            st = need(ns, \"deep_stats\")(f, init, 50, 256, 0)\n            res[name] = st\n            rows.append([name] + [f\"{st[k][0]:+.2f} / {st[k][1]:.2f}\" for k in (0, 9, 49)])\n        sl = res[\"SELU + LeCun\"]\n        ensure(abs(sl[49][0]) < 0.1 and abs(sl[49][1] - 1) < 0.1, \"SELU + LeCun 50 katmanda (0, 1) civarı\")\n        ensure(res[\"SELU + He\"][49][1] > 1.3, \"SELU + He: varyans büyür\")\n        ensure(res[\"tanh + Glorot\"][49][1] < 0.5, \"tanh: aktivasyonlar söner\")\n        R_.table(\"256 genişlikli, 50 katmanlı ağ: ortalama / std (katman 1, 10, 50)\", [\"aktivasyon + başlatma\", \"katman 1\", \"katman 10\", \"katman 50\"], rows,\n                 note=\"SELU'nun λ, α sabitleri E[selu(z)] = 0 ve E[selu²] = 1'i sağlayacak şekilde seçilmiştir; LeCun başlatmayla her katmanın ön-aktivasyonu yine yaklaşık N(0, 1) olur — dağılım katmanlar boyunca kendini düzeltir (öz-normalizasyon). He başlatma (2 kat varyans) bu sabit noktayı bozar; ReLU + He varyansı korur ama ortalama pozitif kayar; tanh + Glorot derinlikte söner.\")\n    R_.test(\"SELU öz-normalizasyonu\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"rrelu\", \"prelu_grads\", \"elu\", \"mish\", \"dead_fraction\")])\n\n    def t():\n        rng = np.random.default_rng(0)\n        z = np.array([-2.0, -0.5, 0.0, 1.5])\n        close(need(ns, \"rrelu\")(z, False, rng), [-2 * (1 / 8 + 1 / 3) / 2, -0.5 * (1 / 8 + 1 / 3) / 2, 0, 1.5], what=\"RReLU test\")\n        tr = ns[\"rrelu\"](np.full(10000, -1.0), True, rng)\n        ensure(-1 / 3 <= tr.min() and tr.max() <= -1 / 8 and abs(tr.mean() + (1 / 8 + 1 / 3) / 2) < 0.01, \"RReLU eğitim: α ~ U(1/8, 1/3)\")\n        zz = rng.normal(size=50)\n        go = rng.normal(size=50)\n        dz, da = need(ns, \"prelu_grads\")(zz, 0.25, go)\n        f = lambda a: np.sum(go * np.where(zz > 0, zz, a * zz))\n        close(da, (f(0.25 + 1e-6) - f(0.25)) / 1e-6, atol=1e-4, what=\"∂/∂α\")\n        close(dz, go * np.where(zz > 0, 1, 0.25), what=\"∂/∂z\")\n        close(need(ns, \"mish\")(np.array([0.0, 1.0, -1.0])), [0, np.tanh(np.log1p(np.e)), -np.tanh(np.log1p(np.exp(-1)))], what=\"Mish\")\n        close(need(ns, \"elu\")(np.array([-1.0, 2.0])), [np.exp(-1) - 1, 2], what=\"ELU\")\n        X = rng.normal(size=(512, 20))\n        W = rng.normal(0, np.sqrt(2 / 20), (20, 100))\n        b = np.zeros(100)\n        rows = []\n        for shift in (0.0, -1.0, -2.0, -4.0):\n            rows.append([shift, f\"{need(ns, 'dead_fraction')(X, W, b + shift * np.sqrt(np.sum(W ** 2, 0))):.2f}\"])\n        ensure(float(rows[0][1]) == 0.0 and float(rows[3][1]) > 0.9, \"ölü nöron oranı bias kaydıkça artar\")\n        R_.table(\"ReLU katmanı: bias, ön-aktivasyon std'sinin kaç katı negatife kaydı\", [\"bias kayması (σ cinsinden)\", \"ölü nöron oranı (512 örneğin hiçbirinde aktif değil)\"], rows,\n                 note=\"Büyük bir gradyan adımı bir ReLU'nun ağırlıklarını tüm eğitim verisi için z < 0 olacak şekilde iterse türevi her yerde 0 olur ve bir daha asla güncellenmez: 'ölü ReLU'. Yüksek öğrenme oranlarında ağın yarısı ölebilir. Sızıntılı varyantlar (α > 0) negatif bölgede de gradyan taşır; Xu ve ark. (2015) bu yüzden hep ReLU'dan iyi buldu. RReLU ayrıca düzenleyici, PReLU büyük veride güçlü.\")\n    R_.test(\"ReLU ailesi ve ölü nöronlar\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        close(need(ns, \"init_std\")(\"glorot\", 300, 100), np.sqrt(2 / 400), what=\"Glorot\")\n        close(ns[\"init_std\"](\"he\", 300, 100), np.sqrt(2 / 300), what=\"He\")\n        close(need(ns, \"uniform_limit\")(\"glorot\", 784, 300), np.sqrt(6 / (784 + 300)), what=\"Glorot uniform: √(6/(fan_in + fan_out))\")\n        sizes = [400, 100, 400, 50, 400]\n        rows = []\n        res = {}\n        for kind in (\"lecun\", \"glorot\"):\n            f, b = need(ns, \"variance_flow\")(sizes, kind)\n            res[kind] = (f, b)\n            rows.append([kind, \" \".join(f\"{v:.2f}\" for v in f), \" \".join(f\"{v:.2f}\" for v in b)])\n        f, b = res[\"lecun\"]\n        ensure(all(abs(v - 1) < 0.1 for v in f) and max(b) > 3, \"fan_in ileri sinyali korur, geri gradyanı korumaz\")\n        fg, bg = res[\"glorot\"]\n        ensure(max(abs(np.log(np.array(fg + bg)))) < max(abs(np.log(np.array(f + b)))), \"fan_avg uzlaşması\")\n        R_.table(\"Doğrusal ağ 400-100-400-50-400: katman başına varyans oranı (1 = korunuyor)\", [\"başlatma\", \"ileri\", \"geri\"], rows,\n                 note=\"Var(çıkış) = fan_in · σ² · Var(giriş): σ² = 1/fan_in ileri sinyali korur. Geri yönde gradyan fan_out ile çarpılır: σ² = 1/fan_out gerekir. Katman genişlikleri farklıyken ikisi aynı anda sağlanamaz; Glorot & Bengio'nun uzlaşması fan_avg'dir — her iki yönde de oranlar 1'e en fazla birkaç kat uzak kalır. ReLU yarıyı sıfırladığı için He 2 çarpanını ekler.\")\n    R_.test(\"başlatma varyansları\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m3_5_init as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir lineer katmanda $z=\sum_{j=1}^{n}w_jx_j$; terimler bağımsız ve sıfır ortalamalıysa

$$
\operatorname{Var}(z)=n\,\operatorname{Var}(w)\,\mathbb E[x^2].
$$

$L$ katman boyunca bu çarpan $L$ kez uygulanır: 1'den biraz büyükse aktivasyonlar patlar, biraz küçükse söner. Çözüm çarpanı 1 yapan ağırlık varyansı:

| başlatma | $\operatorname{Var}(w)$ | hangi aktivasyon |
|---|---|---|
| Xavier/Glorot | $\frac{2}{n_{in}+n_{out}}$ | tanh, lineer |
| Kaiming/He | $\frac{2}{n_{in}}$ | ReLU ailesi |

ReLU sıfır ortalamalı simetrik bir girdinin ikinci momentini yarıya indirir: $\mathbb E[\text{ReLU}(z)^2]=\frac12\operatorname{Var}(z)$. Kaiming'in 2'si bunu telafi eder.

Geri geçişte de aynı hikâye; ek olarak aktivasyonun türevi çarpılır. Sigmoid'in türevi en fazla $0.25$; 10 katmanda gradyan en iyi ihtimalle $10^{-6}$ kat küçülür.

## 1 · Elle

<!-- çözüm -->

Varyans çarpanını hesapla: standart normal ağırlıklarla 100 girdili bir nöron varyansı 100 kat büyütüyor; 5 katmanda $10^{10}$. Xavier ve Kaiming bu çarpanı 1'e çekiyor.

Sonra ReLU'nun etkisini gör: Xavier + ReLU her katmanda ikinci momenti yarıya indiriyor; 10 katmanda bin kat küçülme. Kaiming'in 2 çarpanı tam bunu düzeltiyor.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Lineer katman $z=\sum_{j=1}^{n}w_jx_j$; $w_j,x_j$ bağımsız, ortalaması 0, $\operatorname{Var}(x_j)=1$, $n=100$**

- `v1`: $\operatorname{Var}(w)=1$ ise $\operatorname{Var}(z)$
- `xav`: Xavier $\operatorname{Var}(w)=\frac{2}{n_{in}+n_{out}}$, $n_{in}=n_{out}=100$
- `kai`: Kaiming (ReLU) $\operatorname{Var}(w)=\frac{2}{n_{in}}$

**Derinlik**

- `m2`: $z\sim\mathcal N(0,2)$ ise $\mathbb E[\text{ReLU}(z)^2]$
- `dec`: Xavier + ReLU, 10 katman sonra ikinci moment çarpanı
- `sig`: $\sigma'(0)$
- `sig10`: 10 sigmoid katmanı boyunca $\sigma'$ çarpımı (en iyi durum)

In [ ]:
cevaplar = dict(
    v1=None,
    xav=None,
    kai=None,
    m2=None,
    dec=None,
    sig=None,
    sig10=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"v1":[100,0],"xav":[0.01,0.0001],"kai":[0.02,0.0001],"m2":[1,0.001],"dec":[0.000977,0.00002],"sig":[0.25,0.0001],"sig10":[9.5367e-7,1e-8]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, skaler fonksiyonlar ve türevleri (`math` serbest):

- `relu`, `d_relu` (0'da türev 0 kabul et)
- `leaky_relu(x, a=0.01)`, `d_leaky_relu`
- `sigmoid`, `d_sigmoid` (kararlı sigmoid, M1.4)
- `tanh`, `d_tanh`
- `gelu(x)` — tanh yaklaşımı: $0.5x\big(1+\tanh(\sqrt{2/\pi}(x+0.044715x^3))\big)$; `d_gelu`
- `silu(x)` $=x\,\sigma(x)$, `d_silu`

Kontrol her türevi sayısal türevle, GELU'yu da referans değerlerle karşılaştırır.

In [ ]:
%%writefile stage_b.py
import math


def relu(x): ...
def d_relu(x): ...
def leaky_relu(x, a=0.01): ...
def d_leaky_relu(x, a=0.01): ...
def sigmoid(x): ...
def d_sigmoid(x): ...
def tanh(x): ...
def d_tanh(x): ...
def gelu(x): ...
def d_gelu(x): ...
def silu(x): ...
def d_silu(x): ...


print(gelu(1.0), d_gelu(1.0), silu(-2.0))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

- `xavier_std(fan_in, fan_out)`, `kaiming_std(fan_in)` — standart sapma (varyansın kökü).
- `activation_stds(X, depth, width, std, act, rng)` → uzunluğu `depth` olan liste: her katmandan sonra aktivasyonların std'si. Katman $l$: `W = rng.standard_normal((width, width)) * std`, `H = act(H @ W)`; `act` bir fonksiyon (`np.tanh`, ReLU…). Yalnız katman döngüsü.

Kontrol 30 katmanda üç senaryoyu dener ve tablo üretir: std = 1 (patlama), Xavier + ReLU (sönme), Kaiming + ReLU (kararlı).

In [ ]:
%%writefile stage_c.py
import numpy as np


def xavier_std(fan_in, fan_out):
    ...


def kaiming_std(fan_in):
    ...


def activation_stds(X, depth, width, std, act, rng):
    ...


relu = lambda z: np.maximum(0, z)
X = np.random.default_rng(0).standard_normal((500, 256))
print(activation_stds(X, 10, 256, kaiming_std(256), relu, np.random.default_rng(1))[-1])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch'ta başlatma `torch.nn.init` ile, gözlem **hook**'larla:

```python
h = module.register_forward_hook(lambda m, inp, out: ...)   # ileri
out.register_hook(lambda grad: ...)                           # o tensörün gradyanı
h.remove()
```

- `init_model(model, scheme)` — `scheme ∈ {"kaiming", "xavier"}`; tüm `nn.Linear` ağırlıklarına `kaiming_normal_(…, nonlinearity="relu")` ya da `xavier_normal_`, bias'lara 0.
- `layer_stats(model, X)` → `(act_stds, grad_stds)`; her `nn.Linear` çıktısı için std ve o çıktının gradyanının std'si (kayıp: `model(X).pow(2).mean()`), katman sırasıyla. Hook'ları sonunda kaldır.

In [ ]:
%%writefile stage_d.py
import torch
from torch import nn


def init_model(model, scheme):
    ...


def layer_stats(model, X):
    """→ (act_stds, grad_stds): her nn.Linear çıktısı için, katman sırasıyla."""
    ...


def deep_mlp(width=128, depth=20, act=nn.ReLU):
    layers = []
    for _ in range(depth):
        layers += [nn.Linear(width, width), act()]
    return nn.Sequential(*layers)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Kritik ölçeği deneyerek bul: ağırlıklar $W=\frac{s}{\sqrt{n}}\,G$, $G$ standart normal. ReLU için teori $s^*=\sqrt2$. 64 farklı $s$'yi döngüyle denemek yerine bir batch boyutu yap: aktivasyonlar `(S, N, width)`, her katmanda `H = act(scales[:, None, None] / √width * (H @ G_l.T))`.

- `std_vs_scale(X, Gs, scales, act)` — `Gs: (depth, width, width)` tüm ölçekler için ortak yön; → `(S, depth)` katman başına std. Yalnız katman döngüsü; ölçekler üzerinde döngü yok.

Kontrol tek tek simülasyonla karşılaştırır ve 30 katmandan sonra std'yi en iyi koruyan ölçeği raporlar.

In [ ]:
%%writefile stage_e.py
import torch


def std_vs_scale(X, Gs, scales, act):
    """Gs: (depth, width, width), scales: (S,) → (S, depth). Ölçekler üzerinde döngü yok."""
    ...


g = torch.Generator().manual_seed(0)
X = torch.randn(256, 128, generator=g)
Gs = torch.randn(20, 128, 128, generator=g)
scales = torch.linspace(0.5, 2.5, 64)
print(std_vs_scale(X, Gs, scales, torch.relu)[:, -1])

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · SELU ve öz-normalizasyon: sabit nokta ve 50 katman

**SELU ve öz-normalizasyon** (Klambauer ve ark. 2017): $\mathrm{selu}(z)=\lambda\,z$ ($z>0$), $\lambda\alpha(e^z-1)$ ($z\le0$), $\lambda\approx1.0507$, $\alpha\approx1.6733$. Bu sabitler, girdi $N(0,1)$ iken çıktının ortalamasının 0, varyansının 1 olduğu sabit noktayı sağlar; koşullar: (1) girdiler standartlaştırılmış, (2) tüm gizli katmanlar LeCun normal ($\sigma^2=1/\text{fan\_in}$) ile başlatılmış, (3) mimari sıralı (atlama bağlantısı, dropout yok — alpha dropout hariç, M3.7).

NumPy.

- `selu(z)`.
- `gauss_moments(f, n=20001)` — $z\sim N(0,1)$ için $\mathbb E[f(z)]$ ve $\mathbb E[f(z)^2]$: $[-10,10]$ üzerinde trapez kuralıyla sayısal integral (döngüsüz).
- `deep_stats(act, init, depth, width, seed)` — standart normal girdiyle `depth` katmanlı ağda her katmandan sonra (ortalama, std) listesi; `init ∈ {"lecun", "he", "glorot"}`; yalnız katman döngüsü.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def selu(z):
    ...


def gauss_moments(f, n=20001):
    ...


def deep_stats(act, init, depth, width, seed):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · ReLU ailesi: RReLU, PReLU, ELU, Mish ve ölü ReLU'lar

**ReLU ailesi** (Géron; Xu ve ark. 2015): sızıntılı ReLU, rastgele sızıntılı (RReLU: eğitimde α ~ U(l, u), testte ortalama), parametrik (PReLU: α öğrenilir), ELU, Swish/SiLU ve Mish.

NumPy.

- `rrelu(z, train, rng, l=1/8, u=1/3)` — test modunda $\alpha=(l+u)/2$.
- `prelu_grads(z, alpha, gout)` → `(dz, dalpha)` — $\partial/\partial\alpha$ yalnız negatif girişlerden, toplam.
- `elu(z, a=1.0)`, `mish(z)` ($z\tanh(\mathrm{softplus}(z))$).
- `dead_fraction(X, W, b)` — ReLU katmanında **hiçbir** örnek için aktif olmayan (tüm batch'te z ≤ 0) nöronların oranı.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def rrelu(z, train, rng, l=1 / 8, u=1 / 3):
    ...


def prelu_grads(z, alpha, gout):
    ...


def elu(z, a=1.0):
    ...


def mish(z):
    ...


def dead_fraction(X, W, b):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Glorot, He, LeCun: ileri ve geri varyans dengesi

**Glorot, He, LeCun** (Géron Tablo 11-1): $\text{fan}_{avg}=(\text{fan}_{in}+\text{fan}_{out})/2$.

| başlatma | aktivasyonlar | σ² (normal) |
|---|---|---|
| Glorot | yok, tanh, sigmoid, softmax | $1/\text{fan}_{avg}$ |
| He | ReLU ve türevleri | $2/\text{fan}_{in}$ |
| LeCun | SELU | $1/\text{fan}_{in}$ |

Düzgün dağılımda $r=\sqrt{3\sigma^2}$ ile $U(-r,r)$.

NumPy.

- `init_std(kind, fan_in, fan_out)` ve `uniform_limit(kind, fan_in, fan_out)`.
- `variance_flow(sizes, kind, n=4000, seed=0)` — doğrusal (aktivasyonsuz) ağda ileri sinyal varyansının ve geri gradyan varyansının katman başına oranları: `(fwd, bwd)` listeleri (girdi ve çıkış gradyanı birim varyanslı).

In [ ]:
%%writefile extra_x3.py
import numpy as np


def init_std(kind, fan_in, fan_out):
    ...


def uniform_limit(kind, fan_in, fan_out):
    ...


def variance_flow(sizes, kind, n=4000, seed=0):
    ...

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Klambauer ve ark. (2017): öz-normalize eden sinir ağları

[Klambauer, Unterthiner, Mayr, Hochreiter — Self-Normalizing Neural Networks (NeurIPS 2017)](https://arxiv.org/abs/1706.02515)

Makalenin Şekil 1'ini tekrarla: Fashion-MNIST'te 8, 16 ve 32 katmanlı tam bağlı ağları (a) SELU + LeCun normal, (b) ReLU + He + BatchNorm, (c) ReLU + He ile eğitip eğitim kaybı eğrilerini karşılaştır. x1'deki moment sabit noktasını SELU'nun (μ, ν) eşlemesinin Jacobian'ı üzerinden kararlılık analiziyle (Teorem 1: |özdeğerler| < 1) sayısal olarak doğrula. Alpha dropout'u (M3.7 x1) ekleyip normal dropout'la karşılaştır.


**Adımlar**

1. Moment eşlemesi: (μ, ν) → (E[selu(z)], E[selu(z)²]), z ~ N(μ, ν); Jacobian'ı sonlu farklarla.

**Kabul ölçütleri**

- SELU, BN'siz derin ağlarda ReLU'dan daha düzgün ve hızlı eğitim eğrisi.
- Sabit noktada Jacobian'ın spektral yarıçapı < 1.

## Makale ödevi: He ve ark. (2015): PReLU ve He başlatması

[He, Zhang, Ren, Sun — Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification (ICCV 2015)](https://arxiv.org/abs/1502.01852)

Makale Şekil 3'ün deneyini küçük ölçekte yap: 22 ve 30 katmanlı düz (atlama bağlantısız) CNN'leri CIFAR-10'da Xavier ve He başlatmasıyla eğit; 30 katmanlıda Xavier'in hiç öğrenemediğini, He'nin öğrendiğini göster. PReLU'nun öğrendiği α'ları katman ve kanal başına çiz (makale Tablo 1: ilk katmanlarda büyük, derinde küçük α).


**Adımlar**

1. Kanal başına ve katman başına paylaşılan PReLU'yu karşılaştır.

**Kabul ölçütleri**

- 30 katmanda He başlatma yakınsar, Xavier takılır.
- Öğrenilmiş α'ların derinlikle azaldığı grafik.

## Kaynaklar

- [Glorot & Bengio — Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a.html) — ©
- [He et al. — Delving Deep into Rectifiers (Kaiming init)](https://arxiv.org/abs/1502.01852) — arXiv
- [Karpathy — Zero to Hero 4: Activations & Gradients, BatchNorm](https://karpathy.ai/zero-to-hero.html) — kod MIT; video ©
- [UvA DL Tutorial 3–4: Activation Functions, Optimization and Initialization](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial4/Optimization_and_Initialization.html) — MIT